In [14]:
import numpy as np 

from dataclasses import dataclass, field
from typing import Optional
from basic_graph import solver, topology, visualize
import re

def parse_topology_matrix(text):
    lines = [l.strip() for l in text.splitlines() if l.strip()]

    # Remove ANSI escape sequences (your underline junk)
    ansi_escape = re.compile(r'\x1b\[[0-9;]*m')
    lines = [ansi_escape.sub('', l) for l in lines]

    # First line = header
    header = lines[0].split()
    nodes = header

    edges = []

    for line in lines[1:]:
        parts = line.split()
        row_node = parts[0]
        values = parts[1:]

        for col_node, val in zip(nodes, values):
            if val == 'X':
                continue
            edges.append((row_node, col_node, val))

    return nodes, edges


In [15]:
BANDWIDTH_MAP = {
    "NV12": 300,   # GB/s (one-way)
    "PXB": 28,
    "PIX": 32,
    "PHB": 16,
    "SYS": 10,
}

In [16]:
h100_gpu = """
            GPU0    GPU1    GPU2    GPU3    NIC0    NIC1
--------------------------------------------------------
GPU0         X      NV12    PXB     PXB     PIX     SYS
GPU1        NV12     X      PXB     PXB     PXB     SYS
GPU2        PXB     PXB      X      NV12    PXB     SYS
GPU3        PXB     PXB     NV12     X      PXB     SYS
NIC0        PIX     PXB     PXB     PXB      X      SYS
NIC1        SYS     SYS     SYS     SYS     SYS      X

"""

In [ ]:
def build_topo(nodes,edges,bandwidth_map):
  top_nodes = [] 
  node_str_to_id = {}
  top_edges = []
  for id, node_str in enumerate(nodes): 
    top_nodes.append(topology.Node(id,node_str))
    node_str_to_id[node_str] = id 
  for src_str, dst_str, link_str in edges: 
    src_node = top_nodes[node_str_to_id[src_str]] 
    dst_node = top_nodes[node_str_to_id[dst_str]]
    bandwidth = bandwidth_map[link_str]
    edge = topology.Edge(src_node, dst_node, bandwidth)    
    top_edges.append(edge) 
    
  return topology.Topology(top_nodes, top_edges)
  

In [18]:
nodes,edges = parse_topology_matrix(h100_gpu)

In [19]:
edges

[('GPU0', 'GPU1', 'NV12'),
 ('GPU0', 'GPU2', 'PXB'),
 ('GPU0', 'GPU3', 'PXB'),
 ('GPU0', 'NIC0', 'PIX'),
 ('GPU0', 'NIC1', 'SYS'),
 ('GPU1', 'GPU0', 'NV12'),
 ('GPU1', 'GPU2', 'PXB'),
 ('GPU1', 'GPU3', 'PXB'),
 ('GPU1', 'NIC0', 'PXB'),
 ('GPU1', 'NIC1', 'SYS'),
 ('GPU2', 'GPU0', 'PXB'),
 ('GPU2', 'GPU1', 'PXB'),
 ('GPU2', 'GPU3', 'NV12'),
 ('GPU2', 'NIC0', 'PXB'),
 ('GPU2', 'NIC1', 'SYS'),
 ('GPU3', 'GPU0', 'PXB'),
 ('GPU3', 'GPU1', 'PXB'),
 ('GPU3', 'GPU2', 'NV12'),
 ('GPU3', 'NIC0', 'PXB'),
 ('GPU3', 'NIC1', 'SYS'),
 ('NIC0', 'GPU0', 'PIX'),
 ('NIC0', 'GPU1', 'PXB'),
 ('NIC0', 'GPU2', 'PXB'),
 ('NIC0', 'GPU3', 'PXB'),
 ('NIC0', 'NIC1', 'SYS'),
 ('NIC1', 'GPU0', 'SYS'),
 ('NIC1', 'GPU1', 'SYS'),
 ('NIC1', 'GPU2', 'SYS'),
 ('NIC1', 'GPU3', 'SYS'),
 ('NIC1', 'NIC0', 'SYS')]

In [21]:
top = build_topo(nodes,edges,BANDWIDTH_MAP)

In [22]:
top

Topology(nodes=[Node(id=0, name='GPU0'), Node(id=1, name='GPU1'), Node(id=2, name='GPU2'), Node(id=3, name='GPU3'), Node(id=4, name='NIC0'), Node(id=5, name='NIC1')], edges=[Edge(src=Node(id=0, name='GPU0'), dst=Node(id=1, name='GPU1'), bandwidth=300), Edge(src=Node(id=0, name='GPU0'), dst=Node(id=2, name='GPU2'), bandwidth=28), Edge(src=Node(id=0, name='GPU0'), dst=Node(id=3, name='GPU3'), bandwidth=28), Edge(src=Node(id=0, name='GPU0'), dst=Node(id=4, name='NIC0'), bandwidth=32), Edge(src=Node(id=0, name='GPU0'), dst=Node(id=5, name='NIC1'), bandwidth=10), Edge(src=Node(id=1, name='GPU1'), dst=Node(id=0, name='GPU0'), bandwidth=300), Edge(src=Node(id=1, name='GPU1'), dst=Node(id=2, name='GPU2'), bandwidth=28), Edge(src=Node(id=1, name='GPU1'), dst=Node(id=3, name='GPU3'), bandwidth=28), Edge(src=Node(id=1, name='GPU1'), dst=Node(id=4, name='NIC0'), bandwidth=28), Edge(src=Node(id=1, name='GPU1'), dst=Node(id=5, name='NIC1'), bandwidth=10), Edge(src=Node(id=2, name='GPU2'), dst=Node(i

In [23]:
visualize.show_topology(top, "h100x4_top.html")

'h100x4_top.html'

In [24]:
ring_flow = []
for x in range(4): 
  src = top.nodes[x]
  dst = top.nodes[(x+1)%4]
  quantity = 30 
  ring_flow.append(topology.Flow(src,dst,quantity))

In [26]:
routing = solver.solve(top,ring_flow)

In [27]:
visualize.show_routing(top,routing, "h100x4_ring_routing.html")

'h100x4_ring_routing.html'